# Where We Left Off

In the previous lab we built a safety governor to safely move our hero to a set of desired joint angles, expressed in radians, and we used forward kinematics to track the end effector position in real-time.

# In This Lab

In this lab we will do the opposite: we will move our hero by directly asking for a given end-effector position. Joint angles will be computed using Inverse Kinematics.

# Inverse Kinematics

Let's retrieve the 2D model developed in the first lab. 

We want to find the joint angles $\vartheta_{1}$, $\vartheta_{2}$, $\vartheta_{3}$ given the position of the end effector $P=(x_p,z_p)$. The link lengths $l_1$, $l_2$, $l_3$ are known, since we can measure them.

![2d_model_vs_arm](../lab_1_forward_kinematics/images/2d_model_vs_so-101_arm.png)

Without doing any math, we can immediately see that the problem can have **no solution** or **multiple solutions**.

## No solution

A 2D configuration where the desired end-effector position $\underline{P}_{\text{des}}$ is **out of reach**, i.e., $\|\underline{P}_{\text{des}}\|_2 > \ell_1 + \ell_2 + \ell_3$.

<img src="images/out_of_reach.png" width="600" style="display: block; margin: 0 auto;">

## Multiple solutions

Two distinct 2D configurations reaching the same end-effector pose at $\underline{P}$: elbow-down (left) and elbow-up (right). 

<div style="display: flex; justify-content: center; gap: 20px;">
  <img src="../../lab_1_forward_kinematics/images/2d_model_so-101_arm.png" style="width: 48%; max-width: 600px;">
  <img src="images/elbow_up.png" style="width: 48%; max-width: 600px;">
</div>

## Analytical Solution

We want to invert the forward kinematics equations (found in Lab 1), solving them for the joint angles.

$$\underline{P} = \begin{pmatrix} x \\ z \end{pmatrix} = \begin{pmatrix} l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ l_1 \sin(\vartheta_1) + l_2 \sin(\vartheta_1 + \vartheta_2) + l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) \end{pmatrix}$$

As you can immediately see, we need to find three angles $\vartheta_{1}$, $\vartheta_{2}$, $\vartheta_{3}$ with only two equations. This means that the problem is **kinematically redundant**, i.e., there can be more than one solution, as we previously saw.

This leaves us with an **extra degree of freedom**, i.e., even if we lock one link orientation we can still find a solution to the problem. 

Of course, the desired end-effector position must **not be out of reach**, i.e.,  $\|\underline{P}\|_2 < \ell_1 + \ell_2 + \ell_3$.

To deal with this extra degree of freedom, we decide to **lock the last link orientation** to a given angle $\phi$. We choose the last link because its orientation depends on the most angles, which makes our lives easier.

**Remember:** from the forward kinematics solution we know that $l_3$ orientation depends on its joint angle $\vartheta_3$ and the previous joint angles $\vartheta_2, \vartheta_1$:

$$\underline{P} = \begin{pmatrix} x \\ z \end{pmatrix} = \begin{pmatrix} l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2) + \mathbf{l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3)} \\ l_1 \sin(\vartheta_1) + l_2 \sin(\vartheta_1 + \vartheta_2) + \mathbf{l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3)} \end{pmatrix}$$

This lock adds the following equation:
$$\phi = \vartheta_1 + \vartheta_2 + \vartheta_3$$

Let's visualize its effect:

In [1]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Link lengths and fixed parameters
L1, L2, L3 = 1.2, 1.0, 0.6
TARGET_X, TARGET_Z = 1.6, 0.8
MODE = "elbow-down"

limit = L1 + L2 + L3 + 0.3
circle_theta = np.linspace(0, 2 * np.pi, 100)

def solve_ik(x, z, phi_deg, mode="elbow-down"):
    phi = np.deg2rad(phi_deg)
    
    # 1. Wrist position
    xw = x - L3 * np.cos(phi)
    zw = z - L3 * np.sin(phi)
    
    # 2. Law of cosines for theta2
    D = (xw**2 + zw**2 - L1**2 - L2**2) / (2 * L1 * L2)
    
    if abs(D) > 1.0:
        return None, (xw, zw), D
    
    sign = 1.0 if mode == "elbow-down" else -1.0
    theta2 = np.arctan2(sign * np.sqrt(max(0.0, 1.0 - D**2)), D)
    
    # 3. Solve for theta1
    theta1 = np.arctan2(zw, xw) - np.arctan2(L2 * np.sin(theta2), L1 + L2 * np.cos(theta2))
    
    # 4. Solve for theta3
    theta3 = phi - theta1 - theta2
    
    return (theta1, theta2, theta3), (xw, zw), D

# Initialize FigureWidget
fig = go.FigureWidget()

# 0: Static wrist circle (r = L3) around fixed target
fig.add_trace(go.Scatter(
    x=TARGET_X + L3 * np.cos(circle_theta),
    y=TARGET_Z + L3 * np.sin(circle_theta),
    mode="lines",
    line=dict(color="orange", width=1.5, dash="dash"),
    name="Wrist circle (r = l₃)"
))

# 1, 2, 3: Links 1, 2, 3
fig.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#1f77b4", width=6), name="Link 1 (l₁)"))
fig.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#2ca02c", width=6), name="Link 2 (l₂)"))
fig.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#d62728", width=6), name="Link 3 (l₃)"))

# 4: Joints
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(color="black", size=8),
    showlegend=False
))

# 5: Wrist point
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(color="orange", size=9),
    name="Wrist (x_w, z_w)"
))

# 6: Static target point
fig.add_trace(go.Scatter(
    x=[TARGET_X], y=[TARGET_Z], mode="markers",
    marker=dict(color="red", size=10),
    name="Target (x, z)"
))

# Static 2-link workspace boundary circles
r_outer = L1 + L2
r_inner = abs(L1 - L2)

fig.update_layout(
    width=650,
    height=650,
    xaxis=dict(range=[-limit, limit], title="x", zeroline=True, zerolinecolor="gray", showgrid=True, gridcolor="rgba(128,128,128,0.2)"),
    yaxis=dict(range=[-limit, limit], title="z", scaleanchor="x", scaleratio=1, zeroline=True, zerolinecolor="gray", showgrid=True, gridcolor="rgba(128,128,128,0.2)"),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
    margin=dict(l=40, r=40, t=70, b=40),
    shapes=[
        dict(type="circle", xref="x", yref="y", x0=-r_outer, y0=-r_outer, x1=r_outer, y1=r_outer,
             line=dict(color="lightgray", width=1.5, dash="dot")),
        dict(type="circle", xref="x", yref="y", x0=-r_inner, y0=-r_inner, x1=r_inner, y1=r_inner,
             line=dict(color="lightgray", width=1.5, dash="dot")),
    ]
)

# Only Phi slider remains interactive
phi_deg_w = widgets.FloatSlider(value=24.0, min=-180.0, max=180.0, step=1.0, description="Phi (deg)")

def update_arm(*args):
    phi_deg = phi_deg_w.value
    angles, (xw, zw), D = solve_ik(TARGET_X, TARGET_Z, phi_deg, MODE)
    
    with fig.batch_update():
        fig.data[5].x, fig.data[5].y = [xw], [zw]
        
        if angles is not None:
            t1, t2, t3 = angles
            x0, z0 = 0.0, 0.0
            x1, z1 = L1 * np.cos(t1), L1 * np.sin(t1)
            x2, z2 = x1 + L2 * np.cos(t1 + t2), z1 + L2 * np.sin(t1 + t2)
            x3, z3 = x2 + L3 * np.cos(t1 + t2 + t3), z2 + L3 * np.sin(t1 + t2 + t3)
            
            fig.data[1].x, fig.data[1].y = [x0, x1], [z0, z1]
            fig.data[2].x, fig.data[2].y = [x1, x2], [z1, z2]
            fig.data[3].x, fig.data[3].y = [x2, x3], [z2, z3]
            fig.data[3].line = dict(color="#d62728", width=6, dash="solid")
            fig.data[4].x, fig.data[4].y = [x0, x1, x2], [z0, z1, z2]
            
            fig.layout.title = dict(
                text=(
                    f"Wrist: ({xw:.2f}, {zw:.2f})   |   D = {D:.3f}<br>"
                    f"θ1 = {np.rad2deg(t1):.1f}°,  θ2 = {np.rad2deg(t2):.1f}°,  θ3 = {np.rad2deg(t3):.1f}°"
                ),
                font=dict(color="black", size=14)
            )
        else:
            fig.data[1].x, fig.data[1].y = [], []
            fig.data[2].x, fig.data[2].y = [], []
            fig.data[4].x, fig.data[4].y = [], []
            fig.data[3].x, fig.data[3].y = [xw, TARGET_X], [zw, TARGET_Z]
            fig.data[3].line = dict(color="#d62728", width=3, dash="dash")
            
            fig.layout.title = dict(
                text=f"Wrist out of reach (|D| = {abs(D):.3f} > 1.0)",
                font=dict(color="red", size=14)
            )

phi_deg_w.observe(update_arm, names="value")
update_arm()

display(widgets.VBox([phi_deg_w, fig]))

Now we have one equation per unknown quantity $\vartheta_1, \vartheta_2, \vartheta_3$:

$$\begin{pmatrix} x \\ z \\ \phi \end{pmatrix} = \begin{pmatrix} l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ l_1 \sin(\vartheta_1) + l_2 \sin(\vartheta_1 + \vartheta_2) + l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ \vartheta_1 + \vartheta_2 + \vartheta_3\end{pmatrix}$$

$x, z, \phi$ are known, i.e., they are desired values imposed by us.

### Deliverable 3.1

**Deliverable 3.1:** Find $\vartheta_1, \vartheta_2, \vartheta_3$ starting from $x, z, \phi$

### Solution

#### Substitute $\phi$

Substitute $\phi$ into the $x$ and $z$ equations and move the third link terms ($l_3$) to the left-hand side, since we already know its orientation, to isolate the coordinates of the wrist center $(x_w, z_w)$:

$$x_w = x - l_3 \cos(\phi) = l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2)$$

$$z_w = z - l_3 \sin(\phi) = l_1 \sin(\vartheta_1) + l_2 \sin(\vartheta_1 + \vartheta_2)$$

This reduces the problem to a standard 2-link planar arm reaching the point $(x_w, z_w)$.

#### Find $\vartheta_2$

Square both wrist equations and add them together to eliminate $\vartheta_1$:

$$x_w^2 + z_w^2 = \left(l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2)\right)^2 + \left(l_1 \sin(\vartheta_1) + l_2 \sin(\vartheta_1 + \vartheta_2)\right)^2$$

Expand the right-hand side:

$$x_w^2 + z_w^2 = l_1^2(\cos^2\vartheta_1 + \sin^2\vartheta_1) + l_2^2(\cos^2(\vartheta_1+\vartheta_2) + \sin^2(\vartheta_1+\vartheta_2)) + 2l_1 l_2\left(\cos\vartheta_1\cos(\vartheta_1+\vartheta_2) + \sin\vartheta_1\sin(\vartheta_1+\vartheta_2)\right)$$

Apply the Pythagorean identity ($\cos^2\alpha + \sin^2\alpha = 1$) and the cosine difference identity ($\cos(A - B) = \cos A \cos B + \sin A \sin B$):

$$x_w^2 + z_w^2 = l_1^2 + l_2^2 + 2l_1 l_2 \cos\vartheta_2$$

Solve for $\cos(\vartheta_2)$:

$$\cos(\vartheta_2) = \frac{x_w^2 + z_w^2 - l_1^2 - l_2^2}{2 l_1 l_2} \equiv D$$

Note: A real solution exists only if $\vert{}D\vert{} \le 1$, which means the wrist target is within the arm's reach.

Find $\sin(\vartheta_2)$ using the Pythagorean trigonometric identity: $\sin(\vartheta_2) = \pm\sqrt{1 - D^2}$. The two signs correspond to the elbow-up ($-$) and elbow-down ($+$) configurations. 

Now that we have both $\sin(\vartheta_2)$ and $\cos(\vartheta_2)$ we can use the two-argument arctangent ($\operatorname{atan2}$) to find $\vartheta_2$ preserving the quadrant:

$$\vartheta_2 = \operatorname{atan2}\left(\pm\sqrt{1 - D^2},\; D\right)$$

#### Find $\vartheta_1$

Expand the $\cos(\vartheta_1 + \vartheta_2)$ and $\sin(\vartheta_1 + \vartheta_2)$ terms in the wrist equations using the angle addition formulas ($\cos(\vartheta_1 + \vartheta_2) = \cos\vartheta_1\cos\vartheta_2 - \sin\vartheta_1\sin\vartheta_2$ and $\sin(\vartheta_1 + \vartheta_2) = \sin\vartheta_1\cos\vartheta_2 + \cos\vartheta_1\sin\vartheta_2$):

$$x_w = (l_1 + l_2 \cos\vartheta_2)\cos\vartheta_1 - (l_2 \sin\vartheta_2)\sin\vartheta_1$$$$z_w = (l_2 \sin\vartheta_2)\cos\vartheta_1 + (l_1 + l_2 \cos\vartheta_2)\sin\vartheta_1$$

Let $k_1 = l_1 + l_2 \cos\vartheta_2$ and $k_2 = l_2 \sin\vartheta_2$. The system becomes a linear rotation:

$$\begin{pmatrix} x_w \\ z_w \end{pmatrix} = \begin{pmatrix} k_1 & -k_2 \\ k_2 & k_1 \end{pmatrix} \begin{pmatrix} \cos\vartheta_1 \\ \sin\vartheta_1 \end{pmatrix}$$

Notice that $\vartheta_1$ is the angle of the vector $(x_w, z_w)$ minus the internal angle offset formed by $(k_1, k_2)$:

$$\vartheta_1 = \operatorname{atan2}(z_w, x_w) - \operatorname{atan2}(k_2, k_1)$$

<img src="images/theta_1.png" width="600" style="display: block; margin: 0 auto;">

Substituting $k_1$ and $k_2$ back in:

$$\vartheta_1 = \operatorname{atan2}(z - l_3 \sin\phi,\; x - l_3 \cos\phi) - \operatorname{atan2}(l_2 \sin\vartheta_2,\; l_1 + l_2 \cos\vartheta_2)$$

#### Find $\vartheta_3$

Now that $\vartheta_1$ and $\vartheta_2$ are known, solve the third original equation directly:

$$\vartheta_3 = \phi - \vartheta_1 - \vartheta_2$$

#### All the Equations Solved

Summarizing we have:

$$\begin{aligned} \vartheta_2 &= \operatorname{atan2}\!\left(\pm\sqrt{1 - \left(\frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1 l_2}\right)^2},\; \frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1 l_2}\right) \\ \vartheta_1 &= \operatorname{atan2}(z - l_3\sin\phi,\; x - l_3\cos\phi) - \operatorname{atan2}(l_2 \sin\vartheta_2,\; l_1 + l_2 \cos\vartheta_2) \\ \vartheta_3 &= \phi - \vartheta_1 - \vartheta_2 \end{aligned}$$

Substituting $\vartheta_2$ into the other two equations gives the fully explicit closed form:

$$\begin{aligned} \vartheta_2 &= \operatorname{atan2}\!\left(\pm\sqrt{1 - \left(\frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1 l_2}\right)^2},\; \frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1 l_2}\right) \\ \vartheta_1 &= \operatorname{atan2}(z - l_3\sin\phi,\; x - l_3\cos\phi) - \operatorname{atan2}\!\left(\pm l_2\sqrt{1 - \left(\frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1 l_2}\right)^2},\; l_1 + \frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1}\right) \\ \vartheta_3 &= \phi - \operatorname{atan2}(z - l_3\sin\phi,\; x - l_3\cos\phi) + \operatorname{atan2}\!\left(\pm l_2\sqrt{1 - \left(\frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1 l_2}\right)^2},\; l_1 + \frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1}\right) - \operatorname{atan2}\!\left(\pm\sqrt{1 - \left(\frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1 l_2}\right)^2},\; \frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1 l_2}\right) \end{aligned}$$

### Dealing with $\phi$

Note that we have 3 inputs: $x, z, \phi$. It's our duty to choose the right end effector orientation to be sure that we can reach the target.

To be sure that the target is reachable the following must be true:

$$\vert{}l_1 - l_2\vert{} \le \sqrt{x_w^2 + z_w^2} \le l_1 + l_2$$

Substituting $x_w = x - l_3\cos\phi$ and $z_w = z - l_3\sin\phi$, and squaring all sides gives the reachability condition in terms of $\phi$:

$$(l_1 - l_2)^2 \le (x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 \le (l_1 + l_2)^2$$

Expanding the middle term using $\cos^2\phi + \sin^2\phi = 1$:

$$x^2 + z^2 + l_3^2 - 2l_3(x\cos\phi + z\sin\phi) = r^2 + l_3^2 - 2l_3 r \cos(\phi - \alpha)$$

where $r = \sqrt{x^2 + z^2}$ is the target distance from the base and $\alpha = \operatorname{atan2}(z, x)$ is the target's polar angle. 

Substituting this back into the workspace inequality isolates $\cos(\phi - \alpha)$:

$$\frac{r^2 + l_3^2 - (l_1 + l_2)^2}{2l_3 r} \le \cos(\phi - \alpha) \le \frac{r^2 + l_3^2 - (l_1 - l_2)^2}{2l_3 r}$$

(Note: If the target is beyond the maximum total reach $r > l_1 + l_2 + l_3$, the left bound exceeds $1$ and no $\phi$ can reach the target.)

### Practical Strategies for Selecting $\phi$:

Depending on your application, you can pick $\phi$ from the valid interval above using one of three standard strategies:

- Task-Constrained Orientation: If the task requires a specific approach angle (e.g., $\phi = 0$ for horizontal insertion or $\phi = -\frac{\pi}{2}$ for vertical pick-and-place), check whether $\vert \cos(\vartheta_2) \vert \le 1$. If not, the target cannot be reached at that fixed angle.
- Radial Alignment (Maximum Reach Heuristic): When the task does not constrain the approach angle, the safest choice to guarantee reachability is pointing the third link directly along the vector from the base to the target:$$\phi = \operatorname{atan2}(z, x)$$This maximizes $\cos(\phi - \alpha) = 1$, pulling the wrist center as close to the base as possible ($r_w = \vert{}r - l_3\vert{}$) and maximizing the chance that $\vert \cos(\vartheta_2) \vert \le 1$.
- Interval Search / Nearest-Valid Angle: If a preferred angle $\phi_{\text{pref}}$ violates $\vert \cos(\vartheta_2) \vert \le 1$, clamp $\cos(\phi - \alpha)$ to the nearest boundary of the inequality above and solve for $\phi = \alpha \pm \arccos(\dots)$ to find the closest achievable orientation to $\phi_{\text{pref}}$.

### 2D IK Simulation

Let's simulate it:

In [1]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Link lengths
l1, l2, l3 = 2.0, 2.0, 1.0

def is_reachable(x, z, phi):
    xw = x - l3 * np.cos(phi)
    zw = z - l3 * np.sin(phi)
    D = (xw**2 + zw**2 - l1**2 - l2**2) / (2 * l1 * l2)
    return abs(D) <= 1.0

def auto_compute_phi(x, z):
    # 1. Preferred heuristic: point radially outward from the base
    preferred_phi = np.arctan2(z, x)
    if is_reachable(x, z, preferred_phi):
        return np.rad2deg(preferred_phi)
    
    # 2. Fallback: search angles closest to the preferred radial direction
    offsets = np.linspace(0, np.pi, 180)
    for offset in offsets:
        for sign in (1.0, -1.0):
            candidate = preferred_phi + sign * offset
            if is_reachable(x, z, candidate):
                wrapped = (np.rad2deg(candidate) + 180) % 360 - 180
                return wrapped
                
    return np.rad2deg(preferred_phi)

def solve_ik(x, z, phi_deg, elbow_up=False):
    phi = np.deg2rad(phi_deg)
    
    # 1. Wrist position
    xw = x - l3 * np.cos(phi)
    zw = z - l3 * np.sin(phi)
    
    # 2. Solve for theta_2
    D = (xw**2 + zw**2 - l1**2 - l2**2) / (2 * l1 * l2)
    
    if abs(D) > 1.0:
        return None  # Target out of reach
        
    sign = -1.0 if elbow_up else 1.0
    theta2 = np.arctan2(sign * np.sqrt(1 - D**2), D)
    
    # 3. Solve for theta_1
    theta1 = np.arctan2(zw, xw) - np.arctan2(l2 * np.sin(theta2), l1 + l2 * np.cos(theta2))
    
    # 4. Solve for theta_3
    theta3 = phi - theta1 - theta2
    
    return theta1, theta2, theta3

def forward_kinematics(theta1, theta2, theta3):
    x0, z0 = 0.0, 0.0
    x1 = l1 * np.cos(theta1)
    z1 = l1 * np.sin(theta1)
    x2 = x1 + l2 * np.cos(theta1 + theta2)
    z2 = z1 + l2 * np.sin(theta1 + theta2)
    x3 = x2 + l3 * np.cos(theta1 + theta2 + theta3)
    z3 = z2 + l3 * np.sin(theta1 + theta2 + theta3)
    return [x0, x1, x2, x3], [z0, z1, z2, z3]

# Create FigureWidget
fig = go.FigureWidget()

max_reach = l1 + l2 + l3
circle_theta = np.linspace(0, 2 * np.pi, 200)

# Trace 0: Max Reach Circle
fig.add_trace(go.Scatter(
    x=max_reach * np.cos(circle_theta),
    y=max_reach * np.sin(circle_theta),
    mode='lines',
    line=dict(color='lightgray', dash='dash'),
    name='Max Reach'
))

# Trace 1: Radial Reference Ray (Base -> Target -> Max Reach)
fig.add_trace(go.Scatter(
    x=[0, 0],
    y=[0, 0],
    mode='lines',
    line=dict(color='#2ca02c', width=2, dash='dot'),
    name='Ideal Radial Axis'
))

# Trace 2: End-Effector Heading Indicator
fig.add_trace(go.Scatter(
    x=[0, 0],
    y=[0, 0],
    mode='lines',
    line=dict(color='#ff7f0e', width=3),
    name='Actual φ Heading'
))

# Trace 3: 3R Arm Links
fig.add_trace(go.Scatter(
    x=[0, 0, 0, 0],
    y=[0, 0, 0, 0],
    mode='lines+markers',
    line=dict(color='#1f77b4', width=6),
    marker=dict(size=12, color=['black', '#1f77b4', '#1f77b4', '#ff7f0e']),
    name='3R Arm'
))

# Trace 4: Target Marker
fig.add_trace(go.Scatter(
    x=[3.0],
    y=[2.0],
    mode='markers',
    marker=dict(size=14, symbol='x', color='red'),
    name='Target'
))

fig.update_layout(
    title='3-Link Planar Arm Inverse Kinematics (Radial Alignment)',
    xaxis=dict(range=[-5.5, 5.5], title='X', scaleanchor='y', scaleratio=1),
    yaxis=dict(range=[-5.5, 5.5], title='Z'),
    width=700,
    height=650,
    template='plotly_white'
)

# Interactive controls
x_slider = widgets.FloatSlider(value=3.0, min=-5.0, max=5.0, step=0.1, description='Target X:')
z_slider = widgets.FloatSlider(value=2.0, min=-5.0, max=5.0, step=0.1, description='Target Z:')
auto_phi_checkbox = widgets.Checkbox(value=True, description='Auto-compute φ (Radial Align.)')
phi_slider = widgets.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description='Angle φ (°):', disabled=True)
branch_toggle = widgets.ToggleButtons(options=[('Elbow Down (+)', False), ('Elbow Up (-)', True)], description='Branch:')
status_label = widgets.HTML()

def update_plot(change=None):
    x, z, elbow_up = x_slider.value, z_slider.value, branch_toggle.value
    
    # Ideal radial angle from base (0,0) to target (x,z)
    radial_rad = np.arctan2(z, x)
    radial_deg = np.rad2deg(radial_rad)
    
    if auto_phi_checkbox.value:
        phi_slider.disabled = True
        phi = auto_compute_phi(x, z)
        phi_slider.value = round(phi, 1)
    else:
        phi_slider.disabled = False
        phi = phi_slider.value

    phi_rad = np.deg2rad(phi)
    solution = solve_ik(x, z, phi, elbow_up)
    
    # Angular alignment error (wrapped to [-180, 180])
    align_err = (phi - radial_deg + 180) % 360 - 180
    
    with fig.batch_update():
        # Update Target
        fig.data[4].x = [x]
        fig.data[4].y = [z]
        
        # Update Ideal Radial Ray from origin through target to max_reach
        if np.hypot(x, z) > 1e-6:
            fig.data[1].x = [0.0, max_reach * np.cos(radial_rad)]
            fig.data[1].y = [0.0, max_reach * np.sin(radial_rad)]
        else:
            fig.data[1].x = [0.0, 0.0]
            fig.data[1].y = [0.0, 0.0]
            
        # Update Actual Heading Vector extending 0.8 units past the target
        pointer_len = 0.8
        fig.data[2].x = [x, x + pointer_len * np.cos(phi_rad)]
        fig.data[2].y = [z, z + pointer_len * np.sin(phi_rad)]
        
        if solution is None:
            fig.data[3].line.color = 'rgba(255, 0, 0, 0.3)'
            status_label.value = "<b style='color:red;'>Target Out of Reach</b>"
        else:
            t1, t2, t3 = solution
            xs, zs = forward_kinematics(t1, t2, t3)
            fig.data[3].x = xs
            fig.data[3].y = zs
            fig.data[3].line.color = '#1f77b4'
            
            align_badge = (
                "<span style='color:green;'>Aligned (0.0°)</span>"
                if abs(align_err) < 0.1
                else f"<span style='color:#d95f02;'>Offset: {align_err:+.1f}°</span>"
            )
            
            status_label.value = (
                f"<b>Radial Target:</b> {radial_deg:.1f}° &nbsp;|&nbsp; "
                f"<b>Actual φ:</b> {phi:.1f}° ({align_badge})<br>"
                f"<b>Joints:</b> &vartheta;<sub>1</sub> = {np.rad2deg(t1):.1f}°, "
                f"&vartheta;<sub>2</sub> = {np.rad2deg(t2):.1f}°, "
                f"&vartheta;<sub>3</sub> = {np.rad2deg(t3):.1f}°"
            )

for widget in (x_slider, z_slider, phi_slider, branch_toggle, auto_phi_checkbox):
    widget.observe(update_plot, names='value')

update_plot()
controls = widgets.VBox([
    widgets.HBox([x_slider, z_slider]),
    widgets.HBox([auto_phi_checkbox, phi_slider]),
    widgets.HBox([branch_toggle]),
    status_label
])
display(controls, fig)

FigureWidget({
    'data': [{'line': {'color': 'lightgray', 'dash': 'dash'},
              'mode': 'lines',
              'name': 'Max Reach',
              'type': 'scatter',
              'uid': '621f47fb-a52b-48df-9a59-a28900340a76',
              'x': {'bdata': ('AAAAAAAAFEBd8A+5cv0TQEja9IrL9R' ... 'rL9RNAXfAPuXL9E0AAAAAAAAAUQA=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('AAAAAAAAAAASFg7yMDTEP5MJWgKdMd' ... 'KdMdS/3xUO8jA0xL8Jc9m/rw/WvA=='),
                    'dtype': 'f8'}},
             {'line': {'color': '#2ca02c', 'dash': 'dot', 'width': 2},
              'mode': 'lines',
              'name': 'Ideal Radial Axis',
              'type': 'scatter',
              'uid': '7eebdd1c-dc36-422c-9875-75ab04ab98e1',
              'x': [0.0, 4.160251471689219],
              'y': [0.0, 2.773500981126145]},
             {'line': {'color': '#ff7f0e', 'width': 3},
              'mode': 'lines',
              'name': 'Actual φ Heading',
              'type': 'scat

## Numerical Methods

As you just experienced, deriving the analytical solution of IK is a long and error-prone process, and a closed-form solution does not always exist.

Most of the time, to avoid computing the analytical solution of IK, you treat IK as an optimization problem where you want to minimize the distance between the current position and the desired position, one step at a time, where each step is a small joint movement. Then you use numerical methods to solve the resulting optimization problem.

## The Jacobian Matrix $J(\vartheta)$

The Jacobian matrix describes how small joint variations affect the end effector position. As you can imagine, it is widely used by numerical methods.

### Let's Define It Together

As we already know from Lab 1, the end-effector position is given by the forward kinematics $\mathbf{P} = \mathbf{f}(\boldsymbol{\vartheta})$, where $\boldsymbol{\vartheta} = (\vartheta_1, \dots, \vartheta_n)$ collects the $n$ joint variables. To study the effect of small joint variations, we use derivatives.

The function $\mathbf{f}$ has $m$ components $f_1, \dots, f_m$, one for each task-space coordinate of the end effector. For the simplified 2D model, $f_1 = x$ and $f_2 = z$, so $m = 2$ and $n = 3$. We want to know how each joint variable $\vartheta_j$ affects each end-effector coordinate $f_i$, so we compute every partial derivative $\frac{\partial f_i}{\partial \vartheta_j}$ and arrange them in an $m \times n$ matrix: row $i$ belongs to the coordinate $f_i$, column $j$ to the joint $\vartheta_j$. Column $j$ therefore tells us how the end effector moves when only joint $j$ moves.

Therefore, the Jacobian matrix is defined as follows:

$$\mathbf{J}(\boldsymbol{\vartheta}) = \frac{\partial \mathbf{f}(\boldsymbol{\vartheta})}{\partial \boldsymbol{\vartheta}} = \begin{pmatrix} \frac{\partial f_1}{\partial \vartheta_1} & \cdots & \frac{\partial f_1}{\partial \vartheta_n} \\ \vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial \vartheta_1} & \cdots & \frac{\partial f_m}{\partial \vartheta_n} \end{pmatrix}$$

### Computing It With Nabla

The Jacobian can be easily computed using the Nabla $\nabla$ operator:

$$\mathbf{J}(\boldsymbol{\vartheta}) = \nabla_{\boldsymbol{\vartheta}} \mathbf{f}(\boldsymbol{\vartheta}) = \begin{pmatrix} \nabla_{\boldsymbol{\vartheta}}^T f_1(\boldsymbol{\vartheta}) \\ \nabla_{\boldsymbol{\vartheta}}^T f_2(\boldsymbol{\vartheta}) \\ \vdots \\ \nabla_{\boldsymbol{\vartheta}}^T f_m(\boldsymbol{\vartheta}) \end{pmatrix} = \begin{pmatrix} \frac{\partial f_1}{\partial \vartheta_1} & \frac{\partial f_1}{\partial \vartheta_2} & \cdots & \frac{\partial f_1}{\partial \vartheta_n} \\ \frac{\partial f_2}{\partial \vartheta_1} & \frac{\partial f_2}{\partial \vartheta_2} & \cdots & \frac{\partial f_2}{\partial \vartheta_n} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial \vartheta_1} & \frac{\partial f_m}{\partial \vartheta_2} & \cdots & \frac{\partial f_m}{\partial \vartheta_n} \end{pmatrix}$$

### Deliverable 3.2

**Deliverable 3.2:** Compute the Jacobian for the simplified 2D model of the SO-101 arm using the forward kinematics derived in Lab 1.

### Solution

$$\mathbf{J}(\boldsymbol{\vartheta}) = \begin{pmatrix} -l_1 \sin(\vartheta_1) - l_2 \sin(\vartheta_1 + \vartheta_2) - l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) & -l_2 \sin(\vartheta_1 + \vartheta_2) - l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) & -l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) & l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) & l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) \end{pmatrix}$$

### Describing Movements Using the Jacobian Matrix

For small changes, the Jacobian directly relates joint velocities $\dot{\boldsymbol{\vartheta}}$ to end-effector velocities $\dot{\mathbf{P}}$:

$$\dot{\mathbf{P}} = \mathbf{J}(\boldsymbol{\vartheta})\,\dot{\boldsymbol{\vartheta}}$$

Multiplying both sides by a small time step $\Delta t$ gives a linear approximation of the movement, which can be easily computed on constrained hardware:

$$\Delta \mathbf{P} \approx \mathbf{J}(\boldsymbol{\vartheta})\,\Delta \boldsymbol{\vartheta}$$

## Inverse Kinematics Using the Jacobian Matrix

Since we are talking about inverse kinematics, we want the joint positions starting from the end-effector position, so we need the inverse of the Jacobian:

$$\Delta \boldsymbol{\vartheta} \approx \mathbf{J}^{-1}(\boldsymbol{\vartheta})\,\Delta \mathbf{P}$$

Remember: a matrix inverse exists only when the matrix is square, i.e., $m=n$, and when its determinant is different from zero, i.e., $\det(\mathbf{J}) \neq 0$.

### Deliverable 3.3

**Deliverable 3.3:** Make the Jacobian of the simplified 2D model of the SO-101 arm invertible \
**Hint 1:** $\phi$. \
**Hint 2:** Subtracting one column from another does not change the value of a determinant ($\det(\mathbf{J})$ remains identical).

### Solution

$$\mathbf{J}(\boldsymbol{\vartheta}) = \begin{pmatrix} -l_1 \sin(\vartheta_1) - l_2 \sin(\vartheta_1 + \vartheta_2) - l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) & -l_2 \sin(\vartheta_1 + \vartheta_2) - l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) & -l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) & l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) & l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ 1 & 1 & 1 \end{pmatrix}$$

Let's compute the determinant to check when we can invert it using a couple of tricks to simplify computations. Remember: subtracting one column from another does not change the value of a determinant ($\det(\mathbf{J})$ remains identical).

We first subtract column 2 from column 1:

$$\begin{pmatrix} \big[-l_1 \sin(\vartheta_1) - l_2 \sin(\vartheta_1 + \vartheta_2) - l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3)\big] - \big[-l_2 \sin(\vartheta_1 + \vartheta_2) - l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3)\big] \\ \big[l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3)\big] - \big[l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3)\big] \\ 1 - 1 \end{pmatrix} = \begin{pmatrix} -l_1 \sin(\vartheta_1) \\ l_1 \cos(\vartheta_1) \\ 0 \end{pmatrix}$$

We then subtract column 3 from column 2:

$$\begin{pmatrix} \big[-l_2 \sin(\vartheta_1 + \vartheta_2) - l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3)\big] - \big[-l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3)\big] \\ \big[l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3)\big] - \big[l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3)\big] \\ 1 - 1 \end{pmatrix} = \begin{pmatrix} -l_2 \sin(\vartheta_1 + \vartheta_2) \\ l_2 \cos(\vartheta_1 + \vartheta_2) \\ 0 \end{pmatrix}$$

The determinant becomes:$$\det(\mathbf{J}) = \det \begin{pmatrix} -l_1 \sin(\vartheta_1) & -l_2 \sin(\vartheta_1 + \vartheta_2) & -l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ l_1 \cos(\vartheta_1) & l_2 \cos(\vartheta_1 + \vartheta_2) & l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ 0 & 0 & 1 \end{pmatrix}$$

Applying the Rule of Sarrus we get:

$$\left( \begin{array}{ccc|cc}
-l_1 \sin(\vartheta_1) & -l_2 \sin(\vartheta_1 + \vartheta_2) & -l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) & -l_1 \sin(\vartheta_1) & -l_2 \sin(\vartheta_1 + \vartheta_2) \\
l_1 \cos(\vartheta_1) & l_2 \cos(\vartheta_1 + \vartheta_2) & l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) & l_1 \cos(\vartheta_1) & l_2 \cos(\vartheta_1 + \vartheta_2) \\
0 & 0 & 1 & 0 & 0
\end{array} \right)$$

Add the downward diagonals ($\searrow$) and subtract the upward diagonals ($\nearrow$):

$$\begin{aligned} \det(\mathbf{J}) &= \Big( -l_1 l_2 \sin(\vartheta_1)\cos(\vartheta_1 + \vartheta_2) + 0 + 0 \Big) - \Big( 0 + 0 - l_1 l_2 \sin(\vartheta_1 + \vartheta_2)\cos(\vartheta_1) \Big) \\ &= -l_1 l_2 \sin(\vartheta_1)\cos(\vartheta_1 + \vartheta_2) + l_1 l_2 \sin(\vartheta_1 + \vartheta_2)\cos(\vartheta_1) \\ &= l_1 l_2 \Big( \sin(\vartheta_1 + \vartheta_2)\cos(\vartheta_1) - \cos(\vartheta_1 + \vartheta_2)\sin(\vartheta_1) \Big) \\ &= l_1 l_2 \sin(\vartheta_2) \end{aligned}$$

Impose $\det(J) \neq 0$:

$$\det(\mathbf{J}) = l_1 l_2 \sin(\vartheta_2) \neq 0 \quad \text{for } \vartheta_2 \neq k\pi, \; k \in \mathbb{Z} \quad (l_1, l_2 \neq 0)$$

### What If the Jacobian Inverse Does Not Exist?

Different approximations of the Jacobian inverse have been proposed:

- **Jacobian Transpose** (Gradient Descent Approximation)

  Avoiding matrix inversion altogether by replacing $\mathbf{J}^{-1}$ with the scaled transpose $\alpha \mathbf{J}^T$ (equivalent to gradient descent on the squared error $E(\boldsymbol{\vartheta}) = \frac{1}{2}\Vert{}\Delta \mathbf{P}\Vert{}^2$, since $\nabla_{\boldsymbol{\vartheta}} E = -\mathbf{J}^T \Delta \mathbf{P}$):

  $$
  \Delta \boldsymbol{\vartheta} = -\alpha \nabla_{\boldsymbol{\vartheta}} E(\boldsymbol{\vartheta}) = \alpha \mathbf{J}^T(\boldsymbol{\vartheta})\,\Delta \mathbf{P}
  $$

- **Moore–Penrose Pseudoinverse** (Redundant or Under-Actuated: $m \neq n$)

  For general rectangular Jacobians, the pseudoinverse $\mathbf{J}^+$ finds the minimum-norm joint step ($\min \Vert{}\Delta \boldsymbol{\vartheta}\Vert{}_2^2$) that best achieves the target motion:

  - Redundant arms ($n > m$, more joints than task coordinates):

    $$
    \mathbf{J}^+ = \mathbf{J}^T \left(\mathbf{J}\mathbf{J}^T\right)^{-1} \implies \Delta \boldsymbol{\vartheta} = \mathbf{J}^T \left(\mathbf{J}\mathbf{J}^T\right)^{-1} \Delta \mathbf{P}
    $$

  - Over-constrained arms ($n < m$, fewer joints than task coordinates):

    $$
    \mathbf{J}^+ = \left(\mathbf{J}^T\mathbf{J}\right)^{-1} \mathbf{J}^T \implies \Delta \boldsymbol{\vartheta} = \left(\mathbf{J}^T\mathbf{J}\right)^{-1} \mathbf{J}^T \Delta \mathbf{P}
    $$

  **Limitation:** Still blows up near singularities because the eigenvalues of $\mathbf{J}\mathbf{J}^T$ approach zero, producing huge joint velocities.

- **Damped Least Squares / Levenberg–Marquardt** (Industry Standard: MoveIt / KDL, Pinocchio, MuJoCo, Drake, and PyBullet)

  To stay numerically stable right through singularities, add a damping term $\lambda^2 \mathbf{I}$ (minimizing $\Vert{}\mathbf{J}\Delta \boldsymbol{\vartheta} - \Delta \mathbf{P}\Vert{}_2^2 + \lambda^2 \Vert{}\Delta \boldsymbol{\vartheta}\Vert{}_2^2$):

  $$
  \mathbf{J}^\# = \mathbf{J}^T \left(\mathbf{J}\mathbf{J}^T + \lambda^2 \mathbf{I}\right)^{-1} \implies \Delta \boldsymbol{\vartheta} = \mathbf{J}^T \left(\mathbf{J}\mathbf{J}^T + \lambda^2 \mathbf{I}\right)^{-1} \Delta \mathbf{P}
  $$

  **Why it works:** Even when $\mathbf{J}$ loses rank at a fully stretched arm, $\mathbf{J}\mathbf{J}^T + \lambda^2 \mathbf{I}$ remains strictly positive-definite and invertible.

## Do I Need to Compute Partial Derivatives by Hand?

No. As with forward kinematics, there are automatic ways to compute the Jacobian matrix. For example, CasADi computes the exact symbolic Jacobian `ca.jacobian(f, theta)` automatically from the forward kinematics equations. And in Lab 1 we automatically derived the forward kinematics equations from the URDF using Pinocchio, so we already have the whole pipeline in place. We just need to implement it.

## Simulation and Code Generation with Pinocchio and CasADi (Again)

Here's an updated version of the Pinocchio + CasADi Python script from Lab 1 which also generates the C code for computing the Jacobian in real time on the MCU. 

In [1]:
import os
import numpy as np
import casadi as ca
import pinocchio as pin
from pinocchio import casadi as cpin

URDF_PATH  = "../lab_1_forward_kinematics/so101_new_calib.urdf"
EE_FRAME   = "gripper_frame_link"
ARM_JOINTS = ["shoulder_pan", "shoulder_lift", "elbow_flex",
              "wrist_flex", "wrist_roll"]
OUT_DIR    = "SO101_FK"          # Arduino sketch folder


def load_arm_model():
    """Load the URDF and lock the gripper jaw -> 5-DOF arm model."""
    full = pin.buildModelFromUrdf(URDF_PATH)
    lock = [full.getJointId("gripper")]
    model = pin.buildReducedModel(full, lock, pin.neutral(full))

    names = [model.names[i] for i in range(1, model.njoints)]
    assert names == ARM_JOINTS, f"Unexpected joint order: {names}"
    assert model.nq == 5, f"Expected nq == 5, got {model.nq}"
    assert model.existFrame(EE_FRAME), f"Frame '{EE_FRAME}' not in URDF"
    return model


def build_fk_j_function(model):
    """so101_fk(q) -> (p, R, J)

    q : joint angles (rad), shape 5, URDF convention
    p : tool position in base_link frame (m), shape 3
    R : tool rotation, 3x3
    J : 6D frame Jacobian in LOCAL_WORLD_ALIGNED frame, 6x5
    """
    cmodel = cpin.Model(model)
    cdata = cmodel.createData()

    q = ca.SX.sym("q", model.nq)
    ee_id = model.getFrameId(EE_FRAME)

    cpin.framesForwardKinematics(cmodel, cdata, q)
    J = cpin.computeFrameJacobian(
        cmodel, cdata, q, ee_id, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED
    )

    oMf = cdata.oMf[ee_id]
    p = oMf.translation
    R = oMf.rotation

    return ca.Function("so101_fk", [q], [p, R, J], ["q"], ["p", "R", "J"])

<frozen importlib._bootstrap>:488: RuntimeWarning: to-Python converter for pinocchio::python::DeprecatedBool already registered; second conversion method ignored.


In [2]:
model = load_arm_model()
fk_j = build_fk_j_function(model)

Let's check the shape of the Jacobian

In [3]:
p, R, J = fk_j([0, 0, 0, 0, 0])
print(J.shape)

(6, 5)


A $6 \times 5$ matrix is not square, so the inverse of the Jacobian does not exist. For this reason, we have to use an approximation to build the inverse kinematics function. We decide to use **Damped Least Squares**.

$$
  \mathbf{J}^\# = \mathbf{J}^T \left(\mathbf{J}\mathbf{J}^T + \lambda^2 \mathbf{I}\right)^{-1} \implies \Delta \boldsymbol{\vartheta} \approx \mathbf{J}^T \left(\mathbf{J}\mathbf{J}^T + \lambda^2 \mathbf{I}\right)^{-1} \mathbf{\Delta P}
$$

## Implementing the IK Solver

In [4]:
def build_ik_function(model, fk_j, lam=0.04, w_ori=0.05):
    """so101_ik(q_init, p_des, R_des) -> (q_sol, pos_err_m, ori_err_rad)

    Performs a single 6D Damped Least Squares step inside CasADi:
      - p_des : desired position [m], shape 3
      - R_des : desired rotation matrix, shape 3x3
      - w_ori : orientation weight [m/rad] balancing radians vs. meters
                (also resolves the 5-DOF vs. 6D task-space over-constraint)
    """
    q_init = ca.SX.sym("q_init", 5)
    p_des  = ca.SX.sym("p_des", 3)
    R_des  = ca.SX.sym("R_des", 3, 3)

    q_min = ca.DM(model.lowerPositionLimit)
    q_max = ca.DM(model.upperPositionLimit)

    # Weight matrix: 1.0 for position [m], w_ori for orientation [rad]
    W = ca.diag(ca.vertcat(1.0, 1.0, 1.0, w_ori, w_ori, w_ori))

    # Single DLS step
    p, R, J = fk_j(q_init)

    # 3D position error (m) + 3D world-aligned axis-angle error (rad)
    err_pos = p_des - p
    err_ori = cpin.log3(R_des @ R.T)

    J_w   = W @ J
    err_w = W @ ca.vertcat(err_pos, err_ori)

    # 5x5 Damped Least Squares normal equations
    JTJ = J_w.T @ J_w + (lam**2) * ca.SX.eye(5)
    dq  = ca.solve(JTJ, J_w.T @ err_w)

    q = ca.fmin(ca.fmax(q_init + dq, q_min), q_max)

    # Errors evaluated at the updated configuration
    p_final, R_final, _ = fk_j(q)
    final_pos_err = ca.norm_2(p_des - p_final)
    final_ori_err = ca.norm_2(cpin.log3(R_des @ R_final.T))

    return ca.Function(
        "so101_ik",
        [q_init, p_des, R_des],
        [q, final_pos_err, final_ori_err],
        ["q_init", "p_des", "R_des"],
        ["q_sol", "pos_err", "ori_err"],
    )

Now we can generate the C code for the IK solver:

In [5]:
OUT_DIR = "SO101_IK"

ik = build_ik_function(model, fk_j, lam=0.04, w_ori=0.05)

# C code for the MCU
os.makedirs(OUT_DIR, exist_ok=True)
cg = ca.CodeGenerator("so101_ik.c", {"with_header": True,
                                    "casadi_real": "float",
                                    "casadi_int": "int",
                                    "with_mem": False})
cg.add(ik)
cg.generate(os.path.join(OUT_DIR, ""))

print(f"Wrote {OUT_DIR}/so101_ik.c, so101_ik.h")
print("sz_arg", ik.sz_arg(), "sz_res", ik.sz_res(), "sz_iw", ik.sz_iw(), "sz_w", ik.sz_w())

Wrote SO101_IK/so101_ik.c, so101_ik.h
sz_arg 3 sz_res 3 sz_iw 0 sz_w 157


## Deployment on MCU

### Deliverable 3.4

**Deliverable 3.4:** Write in the next cell a modified version of Deliverable 2.4 that takes a target pose expressed as $x, y, z, roll, pitch, yaw$ as an input from the terminal and uses the generated IK solver, contained in files "so101_ik.c" and "so101_ik.h", to reach the target position. After each commanded movement print the predicted position and orientation errors, as well as the IK solver computation time.

**Test sequence:**

1. Put the robot in the starting pose shown below. This pose should correspond to the zero angles of Deliverable 1.7.

```text
       ╭─────┬────────────────────┬──────╮ ◉╲═════╗   ← moveable claw
       │     │      forearm       │  ▤▤  │╤══╲════╝   ← fixed claw
       ╰┬───┬┴────────────────────┴──────┴┴═══════╝
        │   │
        │   │
        │   │
        │   │   upper arm
        │   │
        │   │
        │   │
   ╭────┴───┴────╮
   │   base ◉    │
   ╰──┬───────┬──╯
  ╭───┴───────┴───╮
  │▓▓▓ C-clamp ▓▓▓│
  ╰───────────────╯
 ═══════════════════════════════════════════════════
                table edge
```

2. Launch the MCU program.
3. Copy paste the current FK readings in the terminal
4. Add to the x-axis position +60 mm
5. Press enter
6. See the robot move
7. Grab the arm
8. Type halt to disable the torque

```text
Your code goes here.
```

### Solution

```text
// SO-101 Cartesian control with IK and a two-state FSM on Arduino VENTUNO Q (STM32H5 MCU)
//
// States:
//   NORMAL - the end effector moves towards a pose typed in the terminal. Each
//            pose command applies a single CasADi IK step (so101_ik) from the
//            current targets, and each servo target is clamped to the joint
//            limits measured in Deliverable 2.3
//   HALT   - torque is disabled on every servo (terminal state)
//
// Transition NORMAL -> HALT when any of these is true:
//   - a servo temperature is above MAX_TEMP_C
//   - a servo current (absolute value) is above MAX_CURRENT_MA
//   - a servo has missed more than MAX_MISSED_READS consecutive readings
//   - setup could not align a servo's goal with its present position
//   - the terminal receives anything other than exactly 6 numbers
//
// Terminal command (one per line, robot base frame):
//   <x> <y> <z> <roll> <pitch> <yaw>
// Position in mm, orientation in radians (ZYX Euler: roll about x, pitch about y,
// yaw about z), the same convention as the status line, so a printed pose can be
// edited and typed back in. The gripper keeps its current target.
// Any other non-empty line (e.g. "halt", "3 0.5", "abc", 5 or 7 values) sends the
// robot to HALT. Each pose command applies one IK step with no accuracy check;
// repeating the same command takes another step towards the pose.
//
// Output at 5 Hz, one line per print:
//   state | joint angles [rad] | gripper | end-effector position and orientation
//         | temperature and current of every servo
// plus targets, IK results, transitions and their reasons when they happen.
//
// Waveshare Bus Servo Adapter (A) in UART mode (jumper A)
// Wiring: D1(TX)->TX, D0(RX)->RX, GND->GND
//
// Requires so101_fk.c/.h and so101_ik.c/.h (CasADi-generated) in the sketch folder.

#include "so101_fk.h"
#include "so101_ik.h"

// ======================= Configuration =======================

const uint32_t BAUD = 1000000;           // SO-101 default after LeRobot setup
const uint8_t  IDS[] = {1, 2, 3, 4, 5, 6};
const uint8_t  NUM_SERVOS = sizeof(IDS);
const uint16_t REPLY_TIMEOUT_US = 5000;
const uint32_t CYCLE_PERIOD_US = 1000000UL / 30;  // write targets + read + FK + guard at 30 Hz
const uint16_t PRINT_PERIOD_MS = 200;             // print status line at 5 Hz
const uint16_t HALT_REPEAT_MS  = 100;             // HALT re-sends torque off at 10 Hz
const uint8_t  MAX_SETUP_ATTEMPTS = 10;           // tries to align goals with present positions

const float TICKS_PER_REV = 4096.0f;
const float RAD_PER_TICK  = 2.0f * PI / TICKS_PER_REV;

// Present_Current LSB in mA (Feetech STS series)
const float MA_PER_CURRENT_UNIT = 6.5f;

// Tick value that maps to angle 0 for each servo (order of IDS)
const int ZERO_TICKS[NUM_SERVOS] = {2093, 2091, 2040, 2026, 2056, 2046};

// Joint limits from Deliverable 2.3, in ticks (order of IDS)
const int MIN_TICKS[NUM_SERVOS] = { 803,  815,  910,  882,  117, 2043};
const int MAX_TICKS[NUM_SERVOS] = {3252, 3244, 3109, 3217, 3936, 3500};

// ----- Halt conditions -----
const int     MAX_TEMP_C       = 60;     // halt if any servo is hotter than this
const long    MAX_CURRENT_MA   = 2700;   // halt if any servo draws more than this
const uint8_t MAX_MISSED_READS = 5;      // halt if any servo misses more than this many reads in a row

const uint8_t INPUT_LINE_LEN = 80;       // max length of a terminal command

// ----- Kinematics -----
// The first NUM_FK_JOINTS entries of IDS map to q[0..4] of so101_fk / so101_ik, in order:
// shoulder_pan, shoulder_lift, elbow_flex, wrist_flex, wrist_roll. ID 6 (gripper) is not used.
const uint8_t NUM_FK_JOINTS = 5;

// Maps servo angle to the model's joint angle: q = SIGN * servo_rad + OFFSET.
// Use -1 where the servo turns opposite to the model's joint axis, and an offset
// (rad) if your zero pose differs from the model's zero pose.
// The IK uses the inverse: servo_rad = SIGN * (q - OFFSET)   (SIGN is +-1)
const float JOINT_SIGN[NUM_FK_JOINTS]   = { 1, 1, 1, 1, 1 };
const float JOINT_OFFSET[NUM_FK_JOINTS] = { 0, 0, 0, 0, 0 };

// ----- Inverse kinematics -----
// One damped-least-squares step is taken per command and applied directly
const uint8_t NUM_POSE_VALUES  = 6;            // x y z roll pitch yaw
const float   INPUT_POS_TO_M   = 0.001f;       // terminal position unit: mm
const float   INPUT_ANG_TO_RAD = 1.0f;         // terminal angle unit: rad

// ======================= FSM types =======================
// Defined before any function: the Arduino builder inserts auto-generated
// function prototypes above the first function, and they need these types.

enum State : uint8_t { STATE_NORMAL, STATE_HALT };

enum FaultType : uint8_t {
  FAULT_OVER_TEMP,
  FAULT_OVER_CURRENT,
  FAULT_NO_REPLY,
  FAULT_SETUP_MISMATCH,
  FAULT_BAD_INPUT
};

struct Fault {
  FaultType type;
  uint8_t   id;      // servo ID (unused for FAULT_BAD_INPUT)
  long      value;   // degC, mA, consecutive missed reads, or setup attempts
};

// ======================= STS3215 protocol =======================

const uint8_t REG_TORQUE_ENABLE       = 0x28;  // 40
const uint8_t REG_GOAL_POSITION       = 0x2A;  // 42, 2 bytes, sign-magnitude
const uint8_t REG_PRESENT_POSITION    = 0x38;  // 56, 2 bytes, sign-magnitude
const uint8_t REG_PRESENT_TEMPERATURE = 0x3F;  // 63, 1 byte, degrees C
const uint8_t REG_PRESENT_CURRENT     = 0x45;  // 69, 2 bytes, sign-magnitude

// One SYNC READ covers Present_Position .. Present_Current (registers 56..70)
const uint8_t READ_START = REG_PRESENT_POSITION;
const uint8_t READ_LEN   = REG_PRESENT_CURRENT + 2 - READ_START;   // 15 bytes

// Offsets of each field inside the block
const uint8_t OFS_POSITION    = REG_PRESENT_POSITION    - READ_START;
const uint8_t OFS_TEMPERATURE = REG_PRESENT_TEMPERATURE - READ_START;
const uint8_t OFS_CURRENT     = REG_PRESENT_CURRENT     - READ_START;

const uint8_t INST_READ       = 0x02;
const uint8_t INST_WRITE      = 0x03;
const uint8_t INST_SYNC_READ  = 0x82;
const uint8_t INST_SYNC_WRITE = 0x83;
const uint8_t BROADCAST_ID    = 0xFE;

uint8_t lastTx[64];                      // largest packet is the SYNC WRITE (26 bytes)
uint8_t lastTxLen = 0;

// Build and send a packet; remember it so we can ignore its echo
void sendPacket(uint8_t id, uint8_t instr, const uint8_t* params, uint8_t n) {
  uint8_t len = n + 2;
  uint8_t sum = id + len + instr;
  lastTxLen = 0;
  lastTx[lastTxLen++] = 0xFF;
  lastTx[lastTxLen++] = 0xFF;
  lastTx[lastTxLen++] = id;
  lastTx[lastTxLen++] = len;
  lastTx[lastTxLen++] = instr;
  for (uint8_t i = 0; i < n; i++) {
    lastTx[lastTxLen++] = params[i];
    sum += params[i];
  }
  lastTx[lastTxLen++] = (uint8_t)~sum;

  while (Serial1.available()) Serial1.read();  // clear stale bytes
  Serial1.write(lastTx, lastTxLen);
  Serial1.flush();
}

// Read one byte with timeout; returns -1 on timeout
int readByte(unsigned long deadline) {
  while ((long)(deadline - micros()) > 0) {
    if (Serial1.available()) return Serial1.read();
  }
  return -1;
}

// Wait for a valid status packet from `id`, skipping our own echo.
// Fills data[] with the payload; returns payload length or -1.
int readStatus(uint8_t id, uint8_t* data, uint8_t maxData) {
  unsigned long deadline = micros() + REPLY_TIMEOUT_US;

  while ((long)(deadline - micros()) > 0) {
    // Find header 0xFF 0xFF
    int b = readByte(deadline);
    if (b != 0xFF) continue;
    b = readByte(deadline);
    if (b != 0xFF) continue;

    int rid = readByte(deadline);
    if (rid == 0xFF) rid = readByte(deadline);   // tolerate extra 0xFF
    int len = readByte(deadline);
    if (rid < 0 || len < 2 || len > maxData + 2) continue;

    uint8_t pkt[4 + READ_LEN + 2];               // header + id + len + err + data + checksum
    if (4 + len > (int)sizeof(pkt)) continue;
    uint8_t p = 0;
    pkt[p++] = 0xFF; pkt[p++] = 0xFF; pkt[p++] = rid; pkt[p++] = len;

    bool ok = true;
    for (int i = 0; i < len; i++) {
      int c = readByte(deadline);
      if (c < 0) { ok = false; break; }
      pkt[p++] = (uint8_t)c;
    }
    if (!ok) return -1;

    // Skip the echo of the packet we just sent
    if (p == lastTxLen && memcmp(pkt, lastTx, p) == 0) continue;

    // Verify checksum
    uint8_t sum = rid + len;
    for (int i = 4; i < p - 1; i++) sum += pkt[i];
    if ((uint8_t)~sum != pkt[p - 1]) continue;
    if (rid != id) continue;

    // pkt[4] = error byte, payload follows
    uint8_t n = len - 2;
    memcpy(data, &pkt[5], n);
    return n;
  }
  return -1;
}

void writeByte(uint8_t id, uint8_t addr, uint8_t val) {
  uint8_t p[] = { addr, val };
  sendPacket(id, INST_WRITE, p, 2);
  uint8_t dummy[4];
  readStatus(id, dummy, sizeof(dummy));  // consume the servo's ack
}

// Writes one byte to every servo at once. Broadcast packets get no reply.
void broadcastWriteByte(uint8_t addr, uint8_t val) {
  uint8_t p[] = { addr, val };
  sendPacket(BROADCAST_ID, INST_WRITE, p, 2);
}

// STS 16-bit values are little-endian with bit 15 as the sign
int decodeSignMag(uint8_t lo, uint8_t hi) {
  uint16_t raw = lo | (hi << 8);
  return (raw & 0x8000) ? -(int)(raw & 0x7FFF) : (int)raw;
}

void encodeSignMag(int v, uint8_t& lo, uint8_t& hi) {
  uint16_t raw = (v < 0) ? (uint16_t)(0x8000 | ((-v) & 0x7FFF)) : (uint16_t)(v & 0x7FFF);
  lo = raw & 0xFF;
  hi = raw >> 8;
}

// Reads a 2-byte register from one servo; returns false if it didn't reply
bool readWord(uint8_t id, uint8_t addr, int& val) {
  uint8_t p[] = { addr, 2 };
  sendPacket(id, INST_READ, p, 2);
  uint8_t data[2];
  if (readStatus(id, data, sizeof(data)) != 2) return false;
  val = decodeSignMag(data[0], data[1]);
  return true;
}

// Sends Goal_Position to all servos with one SYNC WRITE (no reply)
void writeGoalPositions(const int* goal) {
  uint8_t p[2 + 3 * NUM_SERVOS];
  uint8_t k = 0;
  p[k++] = REG_GOAL_POSITION;
  p[k++] = 2;                                // bytes per servo
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    p[k++] = IDS[i];
    encodeSignMag(goal[i], p[k], p[k + 1]);
    k += 2;
  }
  sendPacket(BROADCAST_ID, INST_SYNC_WRITE, p, k);
}

// Reads position, temperature and current of all servos with one SYNC READ.
// replied[i] tells whether servo i answered; its other fields are valid only if so.
// Returns true if all replied.
bool readAllServos(int* ticks, int* tempC, int* currRaw, bool* replied) {
  uint8_t p[2 + NUM_SERVOS];
  p[0] = READ_START;
  p[1] = READ_LEN;                           // bytes per servo
  for (uint8_t i = 0; i < NUM_SERVOS; i++) p[2 + i] = IDS[i];
  sendPacket(BROADCAST_ID, INST_SYNC_READ, p, sizeof(p));

  // Servos reply one after another, in the order of the ID list
  bool ok = true;
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    uint8_t data[READ_LEN];
    int n = readStatus(IDS[i], data, sizeof(data));
    replied[i] = (n == READ_LEN);
    if (!replied[i]) {
      ok = false;
      continue;
    }
    ticks[i]   = decodeSignMag(data[OFS_POSITION], data[OFS_POSITION + 1]);
    tempC[i]   = data[OFS_TEMPERATURE];
    currRaw[i] = decodeSignMag(data[OFS_CURRENT], data[OFS_CURRENT + 1]);
  }
  return ok;
}

// ======================= Shared data =======================

State state = STATE_NORMAL;

// Latest sensor data (valid for servo i only if replied[i] is true)
int  ticks[NUM_SERVOS];
int  tempC[NUM_SERVOS];
int  currRaw[NUM_SERVOS];
bool replied[NUM_SERVOS];

// Goal position of each servo, always inside [MIN_TICKS, MAX_TICKS]
// except at startup, where it equals the present position
int targetTicks[NUM_SERVOS];

// Consecutive missed readings per servo
uint8_t missedReads[NUM_SERVOS] = {0};

// A servo can trip at most two conditions at once (temp + current, or no reply)
const uint8_t MAX_FAULTS = 2 * NUM_SERVOS;
Fault   faults[MAX_FAULTS];
uint8_t numFaults = 0;

// The rejected terminal line and why it was rejected (for the HALT message)
char        badInputText[INPUT_LINE_LEN] = "";
const char* badInputWhy = "";

// Latest kinematics. The last good FK result is kept so a dropped reply
// doesn't blank the output.
float    q[NUM_FK_JOINTS] = {0};
float    eePos[3] = {0, 0, 0};                     // end-effector position [m]
float    eeRot[9] = {1, 0, 0,  0, 1, 0,  0, 0, 1}; // end-effector rotation, column-major
bool     haveFK   = false;   // at least one FK result available
bool     fkFresh  = false;   // FK succeeded on the most recent read
uint32_t fkMicros = 0;       // time taken by the last FK evaluation

// Static work buffers for so101_ik (sizes from the generated header, no malloc)
casadi_int  ikIw[so101_ik_SZ_IW > 0 ? so101_ik_SZ_IW : 1];
casadi_real ikW [so101_ik_SZ_W  > 0 ? so101_ik_SZ_W  : 1];

// ======================= Angle helpers =======================

// Converts a servo's ticks to radians relative to its zero point
float ticksToAngle(uint8_t i, int t) {
  return (t - ZERO_TICKS[i]) * RAD_PER_TICK;
}

// Sets the target of servo i from an angle in radians, clamped to the joint limits
void setTarget(uint8_t i, float rad) {
  float t = ZERO_TICKS[i] + rad / RAD_PER_TICK;
  bool clamped = false;
  if (t < MIN_TICKS[i]) { t = MIN_TICKS[i]; clamped = true; }
  if (t > MAX_TICKS[i]) { t = MAX_TICKS[i]; clamped = true; }
  targetTicks[i] = (int)lroundf(t);

  Serial.print("Target servo ");
  Serial.print(IDS[i]);
  Serial.print(": ");
  Serial.print(ticksToAngle(i, targetTicks[i]), 3);
  Serial.print(" rad (");
  Serial.print(targetTicks[i]);
  Serial.print(" ticks)");
  if (clamped) {
    Serial.print(" - requested ");
    Serial.print(rad, 3);
    Serial.print(" is outside [");
    Serial.print(ticksToAngle(i, MIN_TICKS[i]), 3);
    Serial.print(", ");
    Serial.print(ticksToAngle(i, MAX_TICKS[i]), 3);
    Serial.print("], clamped");
  }
  Serial.println();
}

// ======================= Forward kinematics =======================

// Wraps the CasADi function so101_fk:(q[5]) -> (p[3], R[3x3]).
// p is in meters, in the robot base frame.
// R is stored column-major (CasADi convention): R(row, col) = R[row + 3*col].
bool computeFK(const float* q, float* p, float* R) {
  const casadi_real* arg[so101_fk_SZ_ARG] = { q };
  casadi_real*       res[so101_fk_SZ_RES] = { p, R };
  // SZ_IW and SZ_W are 0, so no work buffers are needed
  return so101_fk(arg, res, NULL, NULL, 0) == 0;
}

inline float Rat(const float* R, int row, int col) { return R[row + 3 * col]; }

// ZYX Euler angles (roll about x, pitch about y, yaw about z), in radians
void rotToRPY(const float* R, float& roll, float& pitch, float& yaw) {
  yaw   = atan2f(Rat(R, 1, 0), Rat(R, 0, 0));
  pitch = atan2f(-Rat(R, 2, 0), sqrtf(Rat(R, 2, 1) * Rat(R, 2, 1) + Rat(R, 2, 2) * Rat(R, 2, 2)));
  roll  = atan2f(Rat(R, 2, 1), Rat(R, 2, 2));
}

// Inverse of rotToRPY: R = Rz(yaw) * Ry(pitch) * Rx(roll), stored column-major
void rpyToRot(float roll, float pitch, float yaw, float* R) {
  float cr = cosf(roll),  sr = sinf(roll);
  float cp = cosf(pitch), sp = sinf(pitch);
  float cy = cosf(yaw),   sy = sinf(yaw);
  // column 0
  R[0] = cy * cp;
  R[1] = sy * cp;
  R[2] = -sp;
  // column 1
  R[3] = cy * sp * sr - sy * cr;
  R[4] = sy * sp * sr + cy * cr;
  R[5] = cp * sr;
  // column 2
  R[6] = cy * sp * cr + sy * sr;
  R[7] = sy * sp * cr - cy * sr;
  R[8] = cp * cr;
}

// Converts the latest reading to joint angles and runs FK.
// FK runs only when all arm joints replied; otherwise the last result is kept.
void updateKinematics() {
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    if (!replied[i]) { fkFresh = false; return; }
  }
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    q[i] = JOINT_SIGN[i] * ticksToAngle(i, ticks[i]) + JOINT_OFFSET[i];
  }
  uint32_t t0 = micros();
  fkFresh  = computeFK(q, eePos, eeRot);
  fkMicros = micros() - t0;
  haveFK   = haveFK || fkFresh;
}

// ======================= Inverse kinematics =======================

// One damped-least-squares step of the CasADi function
// so101_ik:(q_init[5], p_des[3], R_des[3x3]) -> (q_sol[5], pos_err, ori_err).
// R_des is column-major. Errors are evaluated at q_sol.
bool ikStep(const float* qIn, const float* p, const float* R,
            float* qOut, float& posErr, float& oriErr) {
  const casadi_real* arg[so101_ik_SZ_ARG] = { qIn, p, R };
  casadi_real*       res[so101_ik_SZ_RES] = { qOut, &posErr, &oriErr };
  return so101_ik(arg, res, ikIw, ikW, 0) == 0;
}

// Takes ONE damped-least-squares step towards a pose (m, column-major R) and
// sets the arm servo targets to the result. The gripper target is not changed.
void moveToPose(const float* pDes, const float* RDes) {
  // Start from the current targets
  float qStart[NUM_FK_JOINTS];
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    qStart[i] = JOINT_SIGN[i] * ticksToAngle(i, targetTicks[i]) + JOINT_OFFSET[i];
  }

  float    qSol[NUM_FK_JOINTS];
  float    posErr, oriErr;
  uint32_t t0 = micros();
  bool ok = ikStep(qStart, pDes, RDes, qSol, posErr, oriErr);
  uint32_t ikMicros = micros() - t0;

  if (!ok) {
    Serial.println("IK: CasADi evaluation failed, targets unchanged");
    return;
  }

  Serial.print("IK: 1 step, ");
  Serial.print(ikMicros);
  Serial.print(" us, predicted pos err ");
  Serial.print(posErr * 1000.0f, 2);
  Serial.print(" mm, ori err ");
  Serial.print(oriErr, 3);
  Serial.print(" rad, q[rad]");
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    Serial.print(' ');
    Serial.print(qSol[i], 3);
  }
  Serial.println();

  // Model joint angle -> servo angle (inverse of updateKinematics' mapping)
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    setTarget(i, JOINT_SIGN[i] * (qSol[i] - JOINT_OFFSET[i]));
  }
}

// ======================= Terminal input =======================

void printUsage() {
  Serial.println("Usage: \"<x> <y> <z> <roll> <pitch> <yaw>\" in mm and rad, base frame,");
  Serial.println("       ZYX Euler as in the status line. Any other input halts the robot.");
  Serial.println("       Each command applies one IK step from the current targets.");
}

// Records the rejected line and goes to HALT
void haltOnBadInput(const char* text, const char* why) {
  strncpy(badInputText, text, INPUT_LINE_LEN - 1);
  badInputText[INPUT_LINE_LEN - 1] = '\0';
  badInputWhy = why;

  numFaults = 0;
  addFault(FAULT_BAD_INPUT, 0, 0);
  transitionTo(STATE_HALT);
}

void handleCommand(char* line) {
  if (state != STATE_NORMAL) {
    Serial.println("Command ignored: robot is in HALT");
    return;
  }

  // Keep an untouched copy for the HALT message (strtok() modifies line)
  char original[INPUT_LINE_LEN];
  strncpy(original, line, INPUT_LINE_LEN - 1);
  original[INPUT_LINE_LEN - 1] = '\0';

  // Parse all numbers first, so a bad line changes no target
  float v[NUM_POSE_VALUES];
  uint8_t n = 0;
  for (char* tok = strtok(line, " ,;\t"); tok != NULL; tok = strtok(NULL, " ,;\t")) {
    if (n == NUM_POSE_VALUES) {
      haltOnBadInput(original, "too many values");
      return;
    }
    char* end;
    v[n] = strtod(tok, &end);
    if (end == tok || *end != '\0' || !isfinite(v[n])) {
      haltOnBadInput(original, "not a number");
      return;
    }
    n++;
  }

  if (n != NUM_POSE_VALUES) {
    haltOnBadInput(original, "too few values");
    return;
  }

  float pDes[3] = { v[0] * INPUT_POS_TO_M, v[1] * INPUT_POS_TO_M, v[2] * INPUT_POS_TO_M };
  float RDes[9];
  rpyToRot(v[3] * INPUT_ANG_TO_RAD, v[4] * INPUT_ANG_TO_RAD, v[5] * INPUT_ANG_TO_RAD, RDes);

  moveToPose(pDes, RDes);
}

// Collects characters into a line without blocking; runs a command on Enter
void pollTerminal() {
  static char    line[INPUT_LINE_LEN];
  static uint8_t len = 0;
  static bool    overflow = false;

  while (Serial.available()) {
    char c = Serial.read();
    if (c == '\n' || c == '\r') {
      if (overflow) {
        line[len] = '\0';                       // first INPUT_LINE_LEN-1 characters
        if (state == STATE_NORMAL) {
          haltOnBadInput(line, "line too long");
        } else {
          Serial.println("Command ignored: robot is in HALT");
        }
      } else if (len > 0) {
        line[len] = '\0';
        handleCommand(line);
      }
      // Empty lines are ignored: "\r\n" line endings produce one after every command
      len = 0;
      overflow = false;
    } else if (len < INPUT_LINE_LEN - 1) {
      line[len++] = c;
    } else {
      overflow = true;
    }
  }
}

// ======================= Finite State Machine =======================

const char* stateName(State s) {
  switch (s) {
    case STATE_NORMAL: return "NORMAL";
    case STATE_HALT:   return "HALT";
  }
  return "UNKNOWN";
}

void addFault(FaultType type, uint8_t id, long value) {
  if (numFaults < MAX_FAULTS) faults[numFaults++] = { type, id, value };
}

// Guard of the NORMAL -> HALT transition, evaluated on each new reading.
// Records every condition that is currently true; returns true if any.
bool checkHaltConditions() {
  numFaults = 0;
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (!replied[i]) {
      if (missedReads[i] < 255) missedReads[i]++;
      if (missedReads[i] > MAX_MISSED_READS) {
        addFault(FAULT_NO_REPLY, IDS[i], missedReads[i]);
      }
      continue;   // temperature and current are not valid for this servo
    }
    missedReads[i] = 0;

    if (tempC[i] > MAX_TEMP_C) {
      addFault(FAULT_OVER_TEMP, IDS[i], tempC[i]);
    }

    long mA = labs(lroundf(currRaw[i] * MA_PER_CURRENT_UNIT));
    if (mA > MAX_CURRENT_MA) {
      addFault(FAULT_OVER_CURRENT, IDS[i], mA);
    }
  }
  return numFaults > 0;
}

void printFault(const Fault& f) {
  if (f.type == FAULT_BAD_INPUT) {
    Serial.print("  reason: input \"");
    Serial.print(badInputText);
    Serial.print("\" rejected (");
    Serial.print(badInputWhy);
    Serial.print("), expected ");
    Serial.print(NUM_POSE_VALUES);
    Serial.println(" values: x y z roll pitch yaw");
    return;
  }

  Serial.print("  reason: servo ");
  Serial.print(f.id);
  switch (f.type) {
    case FAULT_OVER_TEMP:
      Serial.print(" temperature ");
      Serial.print(f.value);
      Serial.print(" C > ");
      Serial.print(MAX_TEMP_C);
      Serial.println(" C");
      break;
    case FAULT_OVER_CURRENT:
      Serial.print(" current ");
      Serial.print(f.value);
      Serial.print(" mA > ");
      Serial.print(MAX_CURRENT_MA);
      Serial.println(" mA");
      break;
    case FAULT_NO_REPLY:
      Serial.print(" not responding for ");
      Serial.print(f.value);
      Serial.print(" consecutive readings > ");
      Serial.println(MAX_MISSED_READS);
      break;
    case FAULT_SETUP_MISMATCH:
      Serial.print(" goal position could not be aligned with present position after ");
      Serial.print(f.value);
      Serial.println(" attempts");
      break;
    case FAULT_BAD_INPUT:
      break;   // handled above
  }
}

void transitionTo(State next) {
  // Entry action of HALT first, before the (slow) printing
  if (next == STATE_HALT) broadcastWriteByte(REG_TORQUE_ENABLE, 0);

  Serial.print("TRANSITION ");
  Serial.print(stateName(state));
  Serial.print(" -> ");
  Serial.println(stateName(next));
  if (next == STATE_HALT) {
    for (uint8_t k = 0; k < numFaults; k++) printFault(faults[k]);
  }
  state = next;
}

// ----- State behaviours -----

// NORMAL: at 30 Hz send the targets, read the servos, update FK, evaluate the guard
void runNormalState() {
  static unsigned long nextCycle = micros();
  if ((long)(micros() - nextCycle) < 0) return;

  // The deadline advances by exactly one period, so the rate doesn't drift.
  nextCycle += CYCLE_PERIOD_US;
  // If we fell more than a full period behind, resynchronize
  if ((long)(micros() - nextCycle) >= 0) nextCycle = micros() + CYCLE_PERIOD_US;

  writeGoalPositions(targetTicks);
  readAllServos(ticks, tempC, currRaw, replied);
  updateKinematics();

  if (checkHaltConditions()) transitionTo(STATE_HALT);
}

// HALT: keep torque disabled on every servo (re-sent periodically, so a servo
// that missed the first packet or rebooted is still switched off)
void runHaltState() {
  static unsigned long lastSent = millis();
  if (millis() - lastSent >= HALT_REPEAT_MS) {
    lastSent = millis();
    broadcastWriteByte(REG_TORQUE_ENABLE, 0);
  }
}

// ======================= Printing =======================

// Prints a value with a leading space for positives so columns line up
void printAligned(float v, uint8_t decimals) {
  if (v >= 0) Serial.print(' ');
  Serial.print(v, decimals);
}

// Prints an integer right-aligned in a field of `width` characters
void printPadded(long v, uint8_t width) {
  char buf[16];
  snprintf(buf, sizeof(buf), "%*ld", width, v);
  Serial.print(buf);
}

// Prints one status line: state, joints/FK, temperature/current
void printStatus() {
  // State, padded so the columns stay put when it changes
  char st[16];
  snprintf(st, sizeof(st), "%-6s", stateName(state));
  Serial.print("State: ");
  Serial.print(st);

  // Joints, gripper, FK
  Serial.print(" | q[rad] ");
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    printAligned(q[i], 3);
    Serial.print(' ');
  }

  const uint8_t g = NUM_SERVOS - 1;   // gripper index
  Serial.print("| grip ");
  if (!replied[g]) Serial.print("  N/A ");
  else printAligned(ticksToAngle(g, ticks[g]), 3);

  if (haveFK) {
    float roll, pitch, yaw;
    rotToRPY(eeRot, roll, pitch, yaw);

    Serial.print(" | xyz[mm] ");
    for (uint8_t k = 0; k < 3; k++) {
      printAligned(eePos[k] * 1000.0f, 1);
      Serial.print(' ');
    }
    Serial.print("| rpy[rad] ");
    printAligned(roll,  3); Serial.print(' ');
    printAligned(pitch, 3); Serial.print(' ');
    printAligned(yaw,   3);
    Serial.print(" | fk ");
    Serial.print(fkMicros);
    Serial.print("us");
    if (!fkFresh) Serial.print(" (stale)");
  } else {
    Serial.print(" | FK: waiting for all joints");
  }

  // Temperature and current per servo (order of IDS)
  Serial.print(" | T[C] ");
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (!replied[i]) Serial.print(" N/A");
    else printPadded(tempC[i], 4);
  }
  Serial.print(" | I[mA] ");
  long totalmA = 0;
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (!replied[i]) { Serial.print("   N/A"); continue; }
    long mA = lroundf(currRaw[i] * MA_PER_CURRENT_UNIT);
    totalmA += labs(mA);
    printPadded(mA, 6);
  }
  Serial.print(" | total ");
  Serial.print(totalmA);
  Serial.println("mA");
}

// ======================= Setup =======================

// Reads Goal_Position back from every servo; returns the index of the first
// servo whose goal differs from targetTicks, or -1 if all match.
int findGoalMismatch() {
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    int goal;
    if (!readWord(IDS[i], REG_GOAL_POSITION, goal) || goal != targetTicks[i]) return i;
  }
  return -1;
}

void setup() {
  Serial.begin(115200);
  Serial1.begin(BAUD);
  delay(200);

  // Sanity check that the generated functions match what this sketch expects
  const casadi_int* sp = so101_fk_sparsity_in(0);
  if (so101_fk_n_in() != 1 || so101_fk_n_out() != 2 || sp[0] != NUM_FK_JOINTS) {
    Serial.println("ERROR: so101_fk signature does not match NUM_FK_JOINTS");
  }
  const casadi_int* spq = so101_ik_sparsity_in(0);
  const casadi_int* spR = so101_ik_sparsity_in(2);
  if (so101_ik_n_in() != 3 || so101_ik_n_out() != 3 ||
      spq[0] != NUM_FK_JOINTS || spR[0] != 3 || spR[1] != 3) {
    Serial.println("ERROR: so101_ik signature does not match (q[5], p[3], R[3x3])");
  }

  Serial.println("Setup: aligning goal positions with present positions");

  // Before enabling torque, make every goal equal to the present position,
  // so the arm doesn't jump when the servos start holding.
  int mismatch = -1;
  bool aligned = false;
  for (uint8_t attempt = 1; attempt <= MAX_SETUP_ATTEMPTS && !aligned; attempt++) {
    if (attempt > 1) delay(CYCLE_PERIOD_US / 1000);

    bool allReplied = readAllServos(ticks, tempC, currRaw, replied);
    updateKinematics();
    if (checkHaltConditions()) {
      transitionTo(STATE_HALT);
      return;
    }
    if (!allReplied) continue;

    for (uint8_t i = 0; i < NUM_SERVOS; i++) targetTicks[i] = ticks[i];
    writeGoalPositions(targetTicks);

    mismatch = findGoalMismatch();
    aligned  = (mismatch < 0);
  }

  if (!aligned) {
    numFaults = 0;
    addFault(FAULT_SETUP_MISMATCH, mismatch >= 0 ? IDS[mismatch] : 0, MAX_SETUP_ATTEMPTS);
    transitionTo(STATE_HALT);
    return;
  }

  // Goals verified: now it's safe to enable torque
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    writeByte(IDS[i], REG_TORQUE_ENABLE, 1);
  }

  Serial.print("Setup: torque enabled, start angles [rad]");
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    Serial.print(' ');
    Serial.print(ticksToAngle(i, targetTicks[i]), 3);
  }
  Serial.println();

  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (targetTicks[i] < MIN_TICKS[i] || targetTicks[i] > MAX_TICKS[i]) {
      Serial.print("Warning: servo ");
      Serial.print(IDS[i]);
      Serial.println(" starts outside its limits; the next command will bring it inside");
    }
  }
  printUsage();
}

// ======================= Main =======================

void loop() {
  static unsigned long lastPrint = 0;

  pollTerminal();

  switch (state) {
    case STATE_NORMAL: runNormalState(); break;
    case STATE_HALT:   runHaltState();   break;
  }

  // ---- Print the status line at 5 Hz ----
  if (millis() - lastPrint >= PRINT_PERIOD_MS) {
    lastPrint = millis();
    printStatus();
  }
}
```

## Is One Iteration Enough?

### Deliverable 3.5

**Deliverable 3.5:** Fill the table contained in the next cell with the error between the desired pose (current reading +60 mm on x-axis) as input and the one printed by FK on the terminal after motion, using the code developed for Deliverable 3.4.

**Test sequence:**

1. Put the robot in the starting pose shown below. This pose should correspond to the zero angles of Deliverable 1.7.

```text
       ╭─────┬────────────────────┬──────╮ ◉╲═════╗   ← moveable claw
       │     │      forearm       │  ▤▤  │╤══╲════╝   ← fixed claw
       ╰┬───┬┴────────────────────┴──────┴┴═══════╝
        │   │
        │   │
        │   │
        │   │   upper arm
        │   │
        │   │
        │   │
   ╭────┴───┴────╮
   │   base ◉    │
   ╰──┬───────┬──╯
  ╭───┴───────┴───╮
  │▓▓▓ C-clamp ▓▓▓│
  ╰───────────────╯
 ═══════════════════════════════════════════════════
                table edge
```

2. Launch the MCU program.
3. Copy paste the current FK readings in the terminal
4. Add to the x-axis +60 mm to compute the desired pose
5. Copy paste the computed desired pose in the following table (Desired Pose)
6. Press enter
7. See the robot move
8. Copy paste the current FK reading in the following table (Pose Estimated by FK)
9. Grab the arm
10. Type halt to disable the torque

| **Pose**                 | x [mm] | y [mm] | z [mm] | roll [rad] | pitch [rad] | yaw [rad] | **‖Δp‖ [mm]** | **Δθ [rad]** | **IK - Time [us]** |
| :---                     | :---:  | :---:  | :---:  | :---:      | :---:       | :---:     | :---:         | :---:        | :---:              |
| **Desired Pose**         |        |        |        |            |             |           | —             | —            |  —                 |
| **IK Prediction**        | —      | —      | —      | —          | —           | —         |               |              |                    |
| **Pose Estimated by FK** |        |        |        |            |             |           | —             | —            |  —                 |
| **Error (Desired − FK)** |        |        |        |            |             |           |               |              |  —                 |

### Solution

| **Pose**                 | x [mm] | y [mm] | z [mm] | roll [rad] | pitch [rad] | yaw [rad] | **‖Δp‖ [mm]** | **Δθ [rad]**       | **IK - Time [us]** |
| :---                     | :---:  | :---:  | :---:  | :---:      | :---:       | :---:     | :---:         | :---:              | :---:              |
| **Desired Pose**         | 438.3  | -35.0  | 255.2  | 0.716      | 1.487       | 0.615     | —             | —                  |  —                 |
| **IK Prediction**        | —      | —      | —      | —          | —           | —         | 19.84         | 0.099              | 376                |
| **Pose Estimated by FK** | 425.9  | -35.1  | 217.2  | 0.642      | 1.485       | 0.554     | —             | —                  |  —                 |
| **Error (Desired − FK)** | 12.4   | 0.1    | 38.0   | 0.074\*    | 0.002\*     | 0.061\*   |  40.0         | ≈0.014 (≈0.8°)     |  —                 |

\* Per-axis Euler differences are not meaningful near pitch ≈ 90° (gimbal lock), because roll and yaw largely cancel. Use Δθ = ‖log3(R_des · R_FKᵀ)‖ as the orientation error.\
† These measurements were taken on an arm with a missing tooth in the gear of motor 3 (elbow_flex), which is the main cause of the large z-axis error.

### Deliverable 3.6

**Deliverable 3.6:** Try to explain in the next cell what is the cause of the error and the discrepancy with the IK Prediction.

Your explanation goes here.

## Reduce the Error

### Deliverable 3.7

**Deliverable 3.7:** Write in the next cell a modified version of `build_ik_function` that reduces the error.\
**Hint:** do more iterations.

In [3]:
#The modified version of the function goes here

### Solution 3.7

In [6]:
def build_ik_function(model, fk_j, n_iter=15, lam=0.04, w_ori=0.05):
    """so101_ik(q_init, p_des, R_des) -> (q_sol, pos_err_m, ori_err_rad)

    Performs n_iter unrolled 6D Damped Least Squares steps inside CasADi:
      - p_des  : desired position [m], shape 3
      - R_des  : desired rotation matrix, shape 3x3
      - n_iter : number of unrolled DLS iterations
      - w_ori  : orientation weight [m/rad] balancing radians vs. meters
                 (also resolves the 5-DOF vs. 6D task-space over-constraint)
    """
    q_init = ca.SX.sym("q_init", 5)
    p_des  = ca.SX.sym("p_des", 3)
    R_des  = ca.SX.sym("R_des", 3, 3)

    q_min = ca.DM(model.lowerPositionLimit)
    q_max = ca.DM(model.upperPositionLimit)

    # Weight matrix: 1.0 for position [m], w_ori for orientation [rad]
    W = ca.diag(ca.vertcat(1.0, 1.0, 1.0, w_ori, w_ori, w_ori))
    damping = (lam**2) * ca.SX.eye(5)

    # Unrolled DLS iterations
    q = q_init
    for _ in range(n_iter):
        p, R, J = fk_j(q)

        # 3D position error (m) + 3D world-aligned axis-angle error (rad)
        err_pos = p_des - p
        err_ori = cpin.log3(R_des @ R.T)

        J_w   = W @ J
        err_w = W @ ca.vertcat(err_pos, err_ori)

        # 5x5 Damped Least Squares normal equations
        JTJ = J_w.T @ J_w + damping
        dq  = ca.solve(JTJ, J_w.T @ err_w)

        q = ca.fmin(ca.fmax(q + dq, q_min), q_max)

    # Errors evaluated at the final configuration
    p_final, R_final, _ = fk_j(q)
    final_pos_err = ca.norm_2(p_des - p_final)
    final_ori_err = ca.norm_2(cpin.log3(R_des @ R_final.T))

    return ca.Function(
        "so101_ik",
        [q_init, p_des, R_des],
        [q, final_pos_err, final_ori_err],
        ["q_init", "p_des", "R_des"],
        ["q_sol", "pos_err", "ori_err"],
    )

### Generate the C Code Again

Generate the C code for the updated IK solver

In [7]:
OUT_DIR = "SO101_IK_IMPROVED"

ik = build_ik_function(model, fk_j, lam=0.04, w_ori=0.05)

# C code for the MCU
os.makedirs(OUT_DIR, exist_ok=True)
cg = ca.CodeGenerator("so101_ik.c", {"with_header": True,
                                    "casadi_real": "float",
                                    "casadi_int": "int",
                                    "with_mem": False})
cg.add(ik)
cg.generate(os.path.join(OUT_DIR, ""))

print(f"Wrote {OUT_DIR}/so101_ik.c, so101_ik.h")
print("sz_arg", ik.sz_arg(), "sz_res", ik.sz_res(), "sz_iw", ik.sz_iw(), "sz_w", ik.sz_w())

Wrote SO101_IK_IMPROVED/so101_ik.c, so101_ik.h
sz_arg 3 sz_res 3 sz_iw 0 sz_w 170


## Import the New C Files into the Arduino Project

Import the new version of the solver into the Arduino sketch.

### Deliverable 3.8

**Deliverable 3.8:** Fill the table contained in the next cell with the error between the desired pose given as input and the one printed by FK on the terminal after motion, using the code developed for Deliverable 3.4 with the newly generated IK solver.

**Test sequence:**

1. Put the robot in the starting pose shown below. This pose should correspond to the zero angles of Deliverable 1.7.

```text
       ╭─────┬────────────────────┬──────╮ ◉╲═════╗   ← moveable claw
       │     │      forearm       │  ▤▤  │╤══╲════╝   ← fixed claw
       ╰┬───┬┴────────────────────┴──────┴┴═══════╝
        │   │
        │   │
        │   │
        │   │   upper arm
        │   │
        │   │
        │   │
   ╭────┴───┴────╮
   │   base ◉    │
   ╰──┬───────┬──╯
  ╭───┴───────┴───╮
  │▓▓▓ C-clamp ▓▓▓│
  ╰───────────────╯
 ═══════════════════════════════════════════════════
                table edge
```

2. Launch the MCU program
3. Copy paste the desired pose contained in the following table into the terminal
4. Press enter
5. Compute the error

| **Pose**                 | x [mm] | y [mm] | z [mm] | roll [rad] | pitch [rad] | yaw [rad] | **‖Δp‖ [mm]** | **Δθ [rad]** | **IK - Time [us]** |
| :---                     | :---:  | :---:  | :---:  | :---:      | :---:       | :---:     | :---:         | :---:        | :---:              |
| **Desired Pose**         |        |        |        |            |             |           | —             | —            |  —                 |
| **IK Prediction**        | —      | —      | —      | —          | —           | —         |               |              |                    |
| **Pose Estimated by FK** |        |        |        |            |             |           | —             | —            |  —                 |
| **Error (Desired − FK)** |        |        |        |            |             |           |               |              |  —                 |

### Solution

| **Pose**                 | x [mm] | y [mm] | z [mm] | roll [rad] | pitch [rad] | yaw [rad] | **‖Δp‖ [mm]** | **Δθ [rad]**       | **IK - Time [us]** |
| :---                     | :---:  | :---:  | :---:  | :---:      | :---:       | :---:     | :---:         | :---:              | :---:              |
| **Desired Pose**         | 435.7  | -40.5  | 252.3  | 0.632      | 1.483       | 0.514     | —             | —                  | —                  |
| **IK Prediction**        | —      | —      | —      | —          | —           | —         | 0.83          | 0.037              | 3385               |
| **Pose Estimated by FK** | 439.3  | -42.5  | 228.6  | 1.068      | 1.513       | 0.962     | —             | —                  | —                  |
| **Error (Desired − FK)** | -3.6   | 2.0    | 23.7   | -0.436\*   | -0.030\*    | -0.448\*  | 24.1          | ≈0.045 (≈2.6°)     | —                  |

\* Per-axis Euler differences are not meaningful near pitch ≈ 90° (gimbal lock), because roll and yaw largely cancel. Use Δθ = ‖log3(R_des · R_FKᵀ)‖ as the orientation error. \
† These measurements were taken on an arm with a missing tooth in the gear of motor 3 (elbow_flex), which is the main cause of the large z-axis error.

### Deliverable 3.9

**Deliverable 3.9:** Describe how the error changed and try to give an explanation of the change and the discrepancy with the IK Prediction in the next cell.

Your explanation goes here.